In [1]:
import numpy as np
import pandas as pd
import zarr
import torch
import math
from datetime import date, datetime, timedelta
import statsmodels.api as sm
import os
import gc
import re
import shutil
import xarray as xr
import matplotlib.pyplot as plt

In [2]:
INPUT_DIR = "/data_3/scratch/francesco/new_zarr.zarr"
ds = xr.open_zarr(INPUT_DIR, chunks="auto")
ds

/home/francesco/miniconda3/envs/ndvi/lib/python3.11/site-packages/zarr/codecs/vlen_utf8.py:99: UserWarning: The codec `vlen-bytes` is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  return cls(**configuration_parsed)
/home/francesco/miniconda3/envs/ndvi/lib/python3.11/site-packages/zarr/codecs/vlen_utf8.py:44: UserWarning: The codec `vlen-utf8` is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  return cls(**configuration_parsed)


<xarray.Dataset> Size: 12GB
Dimensions:        (last_date_idx: 8, pixel: 1000014, param: 6, date: 3073)
Coordinates:
  * pixel          (pixel) int64 8MB 85668616 85668617 ... 94882985 94882986
  * param          (param) object 48B 'par0' 'par1' 'par2' 'par3' 'par4' 'par5'
  * last_date_idx  (last_date_idx) int64 64B 0 1 2 3 4 5 6 7
  * date           (date) datetime64[ns] 25kB 2017-04-03 ... 2025-08-31
Data variables:
    last_dates     (last_date_idx, pixel) object 64MB dask.array<chunksize=(8, 100), meta=np.ndarray>
    params_upper   (pixel, param) float32 24MB dask.array<chunksize=(100, 6), meta=np.ndarray>
    median_ndvi    (pixel, date) int16 6GB dask.array<chunksize=(100, 365), meta=np.ndarray>
    counter        (date) int16 6kB dask.array<chunksize=(365,), meta=np.ndarray>
    params_lower   (pixel, param) float32 24MB dask.array<chunksize=(100, 6), meta=np.ndarray>
    ndvi           (pixel, date) int16 6GB dask.array<chunksize=(100, 365), meta=np.ndarray>

Spinup years

In [3]:
spinup = 365
subset = ds.isel(pixel=slice(0, 199))

for pixel in np.arange(0, 199):
    # Extract NDVI subset for this pixel
    ndvi_subset = subset["ndvi"].isel(pixel=pixel)[spinup:].load()

    # Create a boolean mask for valid NDVI values
    valid_mask = (ndvi_subset > 0) & (ndvi_subset < 10000)

    # Select the first 7 valid values (indices 0–6)
    ndvi_valid = ndvi_subset.where(valid_mask, drop=True).isel(date=slice(0, 7))

    # Extract the corresponding dates (datetime64[D])
    valid_dates = ndvi_valid["date"].values.astype("datetime64[D]")

    # Make sure we always have 8 entries by padding if needed
    full_dates = np.full(8, np.datetime64("1900-01-01", "D"))  # default filler
    full_dates[:len(valid_dates)] = valid_dates  # fill with available dates
    # Assign the 8 values into last_dates (now always length 8)
    subset["last_dates"].data[:, pixel] = full_dates


In [4]:
def find_min_max_dates_chunk(last_dates_array,first_date,current_date):

    
    start_idx = int((last_dates_array.min() - first_date)  / np.timedelta64(1, 'D'))
    end_idx = int((current_date - first_date)  / np.timedelta64(1, 'D'))

    start_idx = max(0,start_idx)

    return start_idx, end_idx

last_dates_array = subset["last_dates"].astype("datetime64[D]").values

first_date = ds["date"][0].load().astype("datetime64[D]").values

current_date = ds["date"][450].load().astype("datetime64[D]").values


start_idx, end_idx = find_min_max_dates_chunk(last_dates_array,first_date,current_date)

In [ ]:

def smoothing_and_gapfilling(ndvi_arr,median_ndvi_arr,last_array_dates_idx,last_delta,current_delta,deltas_arr, end_idx, pot_outlier_present):

    if pot_outlier_present:

        pot_date_idx = last_array_dates_idx[7]

        pot_ndvi = ndvi_arr[pot_date_idx] / 10000
        pot_median_ndvi = median_ndvi_arr[pot_date_idx] / 10000

        pot_delta = pot_ndvi - pot_median_ndvi

        # perform L1 linear gapfilling

        idx_to_interpolate = np.arange(last_array_dates_idx[6], end_idx +1)
        deltas_L1 = np.array([last_delta, pot_delta,current_delta])
        deltas_L1_idx = np.array([last_array_dates_idx[6],last_array_dates_idx[6], end_idx +1])

        deltas_interpolated = np.interp(idx_to_interpolate,deltas_L1_idx,deltas_L1)

        L1_ndvi = deltas_interpolated + median_ndvi_arr[last_array_dates_idx[6]: end_idx +1] / 10000

        ndvi_arr[last_array_dates_idx[6]: end_idx +1] = L1_ndvi

        # perform L2 smoothing

        idx_to_interpolate = np.arange(last_array_dates_idx[2], last_array_dates_idx[4] +1)

        deltas_L2 = np.concatenate(deltas_arr,np.array([current_delta]))
        deltas_L2_idx = last_array_dates_idx[2:5]

        smoothed_deltas =  sm.nonparametric.lowess(deltas_L2, np.arange(1,9), frac= 1, it=3, return_sorted=False)
        deltas_interpolated = np.interp(idx_to_interpolate,smoothed_deltas,deltas_L2)


        L2_ndvi = deltas_interpolated + median_ndvi_arr[last_array_dates_idx[2]: last_array_dates_idx[4] +1] / 10000

        ndvi_arr[last_array_dates_idx[2]: last_array_dates_idx[4] +1] = L2_ndvi

    else:
        
        # perform L1 linear gapfilling

        idx_to_interpolate = np.arange(last_array_dates_idx[6], end_idx +1)
        deltas_interpolated = np.linspace(last_delta, current_delta,num = len(idx_to_interpolate))

        L1_ndvi = deltas_interpolated + median_ndvi_arr[last_array_dates_idx[6]: end_idx +1] / 10000

        ndvi_arr[last_array_dates_idx[6]: end_idx +1] = L1_ndvi

        # perform L2 smoothing
        idx_to_interpolate = np.arange(last_array_dates_idx[2], last_array_dates_idx[3] +1)

        smoothed_deltas =  sm.nonparametric.lowess(deltas_arr, np.arange(1,8), frac= 1, it=3, return_sorted=False)
        deltas_interpolated = np.linspace(smoothed_deltas[2], smoothed_deltas[3],num = len(idx_to_interpolate))

        L2_ndvi = deltas_interpolated + median_ndvi_arr[last_array_dates_idx[2]: last_array_dates_idx[3] +1] / 10000

        ndvi_arr[last_array_dates_idx[2]: last_array_dates_idx[3] +1] = L2_ndvi
    
    return ndvi_arr

In [ ]:

def continous_ndvi(pixel,last_dates_array,dates,start_idx,end_idx):

    # load all the data

    load_interval = (np.arange(start_idx, end_idx +1))
    ndvi_arr = ds["ndvi"].isel(pixel=pixel, date = load_interval).load().values
    median_ndvi_arr = ds["median_ndvi"].isel(pixel=pixel, date = load_interval).load().values
    start_idx_dates = dates[start_idx]

    start_idx_dates = dates[start_idx]

    last_array_dates_idx = ((last_dates_array_pixel - start_idx_dates)  / np.timedelta64(1, 'D')).astype(int)

    last_date_idx = last_array_dates_idx[6]

    current_ndvi = ndvi_arr[last_date_idx] / 10000
    median_current_value = median_ndvi_arr[last_date_idx] / 10000
    
    last_delta = last_ndvi - last_median_ndvi
    current_delta = current_ndvi - median_current_value
    delta_delta = current_delta - last_delta

    if (current_ndvi > 0) & (current_ndvi > 1):

        # check if its a potential outlier or not
        if (abs(delta_delta) > 0.1) & (abs(current_delta) > 0.1):

            # potential outlier, do not do anyhting and return the original array
            last_dates_array[7] = current_delta
            return ndvi_arr, last_dates_array
        
        else:

            # true obs, check if a potential outlier is pending
            if last_array_dates_idx[7] > 0:

                # potential outlier is present
                pot_date_idx = last_array_dates_idx[7]

                pot_ndvi = ndvi_arr[pot_date_idx] / 10000
                pot_median_ndvi = median_ndvi_arr[pot_date_idx] / 10000

                pot_delta = pot_ndvi - pot_median_ndvi

                if abs(pot_delta) > 0.1:

                    # the pot. outlier was a true value
                    ndvi_arr = smoothing_and_gapfilling(ndvi_arr,last_dates_array,median_ndvi_arr,last_array_dates_idx,
                                                                          last_delta,current_delta,deltas_arr, end_idx, pot_outlier_present = True)
                    
                    # clear any pot. out. pending
                    last_dates_array[7] = np.timedelta64()

                    return ndvi_arr, last_dates_array

                else:

                    # the value was a pot. outlier, ignored
                    ndvi_arr = smoothing_and_gapfilling(ndvi_arr,last_dates_array,median_ndvi_arr,last_array_dates_idx,
                                                                          last_delta,current_delta,deltas_arr, end_idx, pot_outlier_present = False)
                    
                    # clear any pot. out. pending
                    last_dates_array[7] = np.timedelta64()

                    return ndvi_arr, last_dates_array

            else:

                # no pot. outlier
                ndvi_arr = smoothing_and_gapfilling(ndvi_arr,last_dates_array,median_ndvi_arr,last_array_dates_idx,
                                                                          last_delta,current_delta,deltas_arr, end_idx, pot_outlier_present = False)
                
                # clear any pot. out. pending
                last_dates_array[7] = np.timedelta64()

                return ndvi_arr, last_dates_array

    else:
             
        # no obs, estimate the current ndvi value
        tau = last_date_idx - end_idx
        estimated_delta = last_delta * np.exp((-tau/45))
        ndvi_arr[-1] = estimated_delta + median_current_value

        return ndvi_arr, last_dates_array

In [9]:
# set initial condition
first_date = ds["date"][0].load().astype("datetime64[D]").values
first_date = np.datetime64(first_date, "D")
pixel = 1
last_dates_array = np.array(subset["last_dates"].isel(pixel=1).load().values, dtype="datetime64[D]")

# run the continous ingestion
for i in np.arange(20,100):

    current_date = np.array(ds["date"][spinup + i].load().values, dtype="datetime64[D]").item()
    last_dates_array = continous_ndvi(pixel,current_date,last_dates_array,first_date)
    last_dates_array = np.array(last_dates_array, dtype="datetime64[D]")
    print(last_dates_array)


['2018-04-22' '2018-05-25' '2018-06-14' '2018-06-16' '2018-06-19'
 '2018-06-21' '2018-06-26' '1900-01-01']
['2018-04-22' '2018-05-25' '2018-06-14' '2018-06-16' '2018-06-19'
 '2018-06-21' '2018-06-26' '1900-01-01']
['2018-04-22' '2018-05-25' '2018-06-14' '2018-06-16' '2018-06-19'
 '2018-06-21' '2018-06-26' '1900-01-01']
['2018-04-22' '2018-05-25' '2018-06-14' '2018-06-16' '2018-06-19'
 '2018-06-21' '2018-06-26' '1900-01-01']
['2018-04-22' '2018-05-25' '2018-06-14' '2018-06-16' '2018-06-19'
 '2018-06-21' '2018-06-26' '1900-01-01']
['2018-04-22' '2018-05-25' '2018-06-14' '2018-06-16' '2018-06-19'
 '2018-06-21' '2018-06-26' '1900-01-01']
['2018-04-22' '2018-05-25' '2018-06-14' '2018-06-16' '2018-06-19'
 '2018-06-21' '2018-06-26' '1900-01-01']
['2018-04-22' '2018-05-25' '2018-06-14' '2018-06-16' '2018-06-19'
 '2018-06-21' '2018-06-26' '1900-01-01']
['2018-04-22' '2018-05-25' '2018-06-14' '2018-06-16' '2018-06-19'
 '2018-06-21' '2018-06-26' '1900-01-01']
['2018-04-22' '2018-05-25' '2018-06-1

ValueError: dimensions () must have the same length as the number of data dimensions, ndim=1